In [1]:
import sys, os, re, time, warnings, ast
from collections import Counter

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from atproto import Client as BskyClient

warnings.filterwarnings("ignore")

In [4]:
import os, sys
config_path = os.path.join("..", "..", "A. Lectures", "Lecture 1 Social network analysis")
sys.path.insert(0, config_path)
from bsky_config_students import BLUESKY_USERNAME, BLUESKY_APP_PASSWORD

client = BskyClient()
client.login(BLUESKY_USERNAME, BLUESKY_APP_PASSWORD)
print(f"Authenticated as: {BLUESKY_USERNAME}")

Authenticated as: vermeulen-anna.bsky.social


In [ ]:
ELECTIONS = {
    "US_2024": {
        "since"        : "2024-07-05T00:00:00Z",
        "until"        : "2024-11-04T23:59:59Z",
        "election_date": "2024-11-05",
        "hashtags": [
            # ── Core election ──────────────────────────────────────────────
            "#Election2024", "#USElection2024", "#ElectionDay", "#Vote2024",
            "#Presidential2024", "#PresidentialElection", "#ElectionNight",
            "#AmericaDecides", "#Decision2024", "#VoterRegistration",
            "#EarlyVoting", "#MailInVoting", "#ElectionIntegrity",
            "#Battleground2024", "#SwingState",

            # ── Trump / Republican ─────────────────────────────────────────
            "#Trump2024", "#TrumpVance", "#VoteTrump", "#Trump",
            "#DonaldTrump", "#MAGA", "#MAGA2024", "#AmericaFirst",
            "#TrumpRally", "#Republicans", "#GOP", "#Republican",
            "#JDVance", "#Vance2024", "#VanceVP",
            "#RNC2024", "#RepublicanConvention",
            "#Project2025", "#TrumpDebate",

            # ── Harris / Democrat ──────────────────────────────────────────
            "#Harris2024", "#KamalaHarris2024", "#KamalaHarris",
            "#HarrisWalz", "#VoteHarris", "#VoteBlue", "#VoteKamala",
            "#Kamala2024", "#Kamala", "#Harris",
            "#TimWalz", "#Walz2024", "#WalzVP",
            "#DNC2024", "#DemConvention", "#DemocraticConvention",
            "#WeAreNotGoingBack", "#WinWithKamala",
            "#Democrats", "#Democrat",

            # ── Biden exit ─────────────────────────────────────────────────
            "#Biden", "#JoeBiden", "#BidenDropsOut", "#BidenOut",
            "#BidenWithdraws", "#Bidenomics",

            # ── Debates & key moments ──────────────────────────────────────
            "#PresidentialDebate", "#Debate2024", "#VPDebate",
            "#TrumpHarrisDebate", "#DebateNight",

           
        ],
    },
}

ACTIVE_ELECTION   = "US_2024"
cfg               = ELECTIONS[ACTIVE_ELECTION]
MIN_TEXT_LENGTH   = 30

print(f"Active election : {ACTIVE_ELECTION}")
print(f"Window          : {cfg['since'][:10]}  to  {cfg['until'][:10]}")
print(f"Hashtags        : {len(cfg['hashtags'])}")
print(f"Limit           : geen (alle beschikbare posts worden opgehaald)")

In [ ]:
import re, time

# ── Text cleaning ──────────────────────────────────────────────────────────────
def clean_text(text):
    """Normalise post text: strip leading/trailing URLs, collapse whitespace."""
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── Quality filters ────────────────────────────────────────────────────────────
NEWS_BOT_PATTERNS = [
    "forbes", "guardian", "bbc", "cnn", "reuters", "apnews",
    "skynews", "france24", "euronews", "washingtonpost", "nytimes",
    "independent", "trtworld", "straits_times", "abc", "news-feed",
    "theguardian", "huffpost", "politico", "axios",
]

def is_news_bot(handle):
    return any(p in handle.lower() for p in NEWS_BOT_PATTERNS)

def has_real_text(text):
    clean = re.sub(r"#\w+|@\w+", "", text).strip()
    return len(clean) >= MIN_TEXT_LENGTH


# ── Candidate labelling ────────────────────────────────────────────────────────
TRUMP_KW  = ["trump", "maga", "donald", "republican", "gop", "conservative", "vance"]
HARRIS_KW = ["harris", "kamala", "democrat", "democratic", "walz", "liberal", "voteblue"]

def label_candidate(row):
    text = row["text"].lower()
    score_a = sum(k in text for k in TRUMP_KW)
    score_b = sum(k in text for k in HARRIS_KW)
    if score_a > score_b: return "CandidateA"
    if score_b > score_a: return "CandidateB"
    return "Neutral"


# ── Post search (geen limiet — pagineert tot geen resultaten meer) ─────────────
def search_posts(client, query, since=None, until=None):
    collected, cursor = [], None
    while True:
        params = {"q": query, "limit": 100, "sort": "latest"}
        if cursor: params["cursor"] = cursor
        if since:  params["since"]  = since
        if until:  params["until"]  = until
        try:
            resp = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"  API error: {e}"); break
        if not resp.posts: break
        for post in resp.posts:
            rec  = post.record
            raw  = getattr(rec, "text", "") or ""
            text = clean_text(raw)
            if not text:
                continue
            collected.append({
                "uri"      : post.uri,
                "author"   : post.author.handle,
                "display"  : getattr(post.author, "display_name", "") or "",
                "text"     : text,
                "timestamp": getattr(rec, "created_at", "") or "",
                "likes"    : post.like_count   or 0,
                "reposts"  : post.repost_count or 0,
                "replies"  : post.reply_count  or 0,
                "mentions" : re.findall(r"@([\w.\-]+)", raw),
                "is_reply" : bool(getattr(rec, "reply", None)),
                "post_type": "post",
                "query"    : query,
            })
        cursor = getattr(resp, "cursor", None)
        if not cursor: break
        time.sleep(0.5)
    return collected


# ── Thread reply fetcher ───────────────────────────────────────────────────────
def fetch_replies(client, uri, depth=3):
    try:
        resp = client.app.bsky.feed.get_post_thread({"uri": uri, "depth": depth})
    except Exception as e:
        print(f"  Thread error: {e}")
        return []

    replies = []

    def walk(node):
        if node is None:
            return
        post = getattr(node, "post", None)
        if post is None:
            return
        rec = getattr(post, "record", None)
        raw = getattr(rec, "text", "") or ""
        text = clean_text(raw)
        if text and not is_news_bot(post.author.handle):
            replies.append({
                "uri"      : post.uri,
                "author"   : post.author.handle,
                "display"  : getattr(post.author, "display_name", "") or "",
                "text"     : text,
                "timestamp": getattr(rec, "created_at", "") or "",
                "likes"    : post.like_count   or 0,
                "reposts"  : post.repost_count or 0,
                "replies"  : post.reply_count  or 0,
                "mentions" : re.findall(r"@([\w.\-]+)", raw),
                "is_reply" : True,
                "post_type": "reply",
                "query"    : "thread",
            })
        for child in (getattr(node, "replies", None) or []):
            walk(child)

    walk(getattr(resp, "thread", None))
    return replies


print("All helper functions defined: clean_text, is_news_bot, has_real_text, label_candidate, search_posts, fetch_replies")

In [ ]:
# ── Clean & sort the scraped CSV ─────────────────────────────────────────────
# Filters out-of-window posts, sorts chronologically, reorders columns.
_csv = f"./Data/Bluesky/bsky_{ACTIVE_ELECTION}_posts.csv"

if not os.path.exists(_csv):
    print("No CSV found — run the scraping cell first.")
else:
    _df = pd.read_csv(_csv)
    _before = len(_df)

    _df["datetime"] = pd.to_datetime(_df["timestamp"], format="ISO8601", utc=True)

    # Filter within election window
    _df = _df[
        (_df["datetime"] >= cfg["since"]) &
        (_df["datetime"] <= cfg["until"])
    ]

    # Sort chronologically
    _df = _df.sort_values("datetime").reset_index(drop=True)

    # Logical column order
    _cols = ["timestamp", "author", "display", "text", "candidate", "post_type",
             "likes", "reposts", "replies", "mentions", "is_reply", "query", "uri", "election"]
    _cols = [c for c in _cols if c in _df.columns]
    _df = _df[_cols]

    _df.to_csv(_csv, index=False)

    print(f"CSV cleaned: {len(_df):,} posts  ({_before - len(_df)} out-of-window removed)")
    print(f"Date range : {_df.timestamp.iloc[0][:10]}  →  {_df.timestamp.iloc[-1][:10]}")
    print(f"Columns    : {_cols}")


In [ ]:
# label_candidate is defined in cell t07 above
print("label_candidate ready ✓")
